# 4 — Full 360-episode Stage 2 run
Launches one detached, resumable process. It loads one horizon at a time and runs every episode serially on the frozen A100. Do not launch another worker.


In [ ]:
import os, subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; PY=Path.home()/"venv-stage1-id/bin/python"; OUT=Path.home()/"stage2"; GPU=(Path.home()/"stage2_gpu.txt").read_text().strip()
pidfile=OUT/"stage2_full.pid"; log=OUT/"stage2_full.log"
if pidfile.exists():
    old=int(pidfile.read_text())
    try: os.kill(old,0); raise SystemExit(f"STOP: Stage 2 worker {old} is already alive")
    except ProcessLookupError: pass
env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONUNBUFFERED":"1"})
cmd=[str(PY),"-u","-m","async_vla_benchmark.scripts.run_stage2","--config",str(R/"async_vla_benchmark/configs/stage2.yaml"),"--manifest",str(OUT/"stage2_local_sensitivity_manifest.csv"),"--output-dir",str(OUT),"--resume","--verbose"]
fh=open(log,"ab"); proc=subprocess.Popen(cmd,cwd=R,env=env,stdout=fh,stderr=subprocess.STDOUT,start_new_session=True); pidfile.write_text(str(proc.pid)); print("launched Stage 2",proc.pid,log)


In [ ]:
# Re-run this cell after reconnecting; progress comes from durable episode artifacts.
import csv
planned={r['run_id'] for r in csv.DictReader(open(OUT/"stage2_local_sensitivity_manifest.csv"))}; done={p.stem for p in (OUT/"episodes").glob("*.json")}; complete=len(planned & done)
alive=False
if pidfile.exists():
    try: os.kill(int(pidfile.read_text()),0); alive=True
    except (ProcessLookupError,PermissionError): pass
print(f"episodes {complete}/360; remaining {360-complete}; worker alive={alive}")
print(''.join(log.read_text().splitlines(True)[-15:]) if log.exists() else '(no log)')
print("Copy ~/stage2 off-machine regularly. If the worker dies, rerun the launch cell; --resume skips completed episodes.")
